In [7]:
import xarray as xr
import numpy as np
import glob

In [2]:
start = 1979
end = 2024

In [3]:
### CPC
ds = xr.open_dataset("../processed_data/cpc/cpc_day_precip_5x5.nc")
ds_rx1day = ds.groupby(ds.time.dt.year).max(dim = "time")

ds_rx1day.to_netcdf("../processed_data/cpc/cpc_rx1day_5x5.nc")

ds_rx1day = ds_rx1day.sel(year = slice(start, end))
ds_trend = ds_rx1day.polyfit(dim = "year", deg = 1)
ds_trend.to_netcdf("../processed_data/cpc_trends/cpc_rx1day_"+str(start)+"_"+str(end)+"_5x5_trend.nc")

In [4]:
### MSWEP
ds = xr.open_dataset("../processed_data/mswep/mswep_day_precip_5x5.nc")
ds_rx1day = ds.groupby(ds.time.dt.year).max(dim = "time")

ds_rx1day.to_netcdf("../processed_data/mswep/mswep_rx1day_5x5.nc")

ds_rx1day = ds_rx1day.sel(year = slice(start, end))
ds_trend = ds_rx1day.polyfit(dim = "year", deg = 1)
ds_trend.to_netcdf("../processed_data/mswep_trends/mswep_rx1day_"+str(start)+"_"+str(end)+"_5x5_trend.nc")

In [5]:
### MESACLIP

sim_keys = [".001.", ".002.", ".003.", ".004.", ".005.", ".006.", ".007.", ".009.", ".010"]

for sim in sim_keys:
    ds = xr.open_dataset("../processed_data/mesaclip/mesaclip_day_precip_"+sim.replace(".", "")+"_5x5.nc")
    ## convert to mm from m/s
    ds["pr"] = ds.pr*60*60*24*1000

    ds_rx1day = ds.groupby(ds.time.dt.year).max(dim = "time")
    ds_rx1day.to_netcdf("../processed_data/mesaclip/mesaclip_rx1day_"+sim.replace(".", "")+"_5x5.nc")

    ds_rx1day = ds_rx1day.sel(year = slice(start, end))
    ds_trend = ds_rx1day.polyfit(dim = "year", deg = 1)
    ds_trend.to_netcdf("../processed_data/mesaclip_trends/mesaclip_rx1day_"+sim.replace(".", "")+\
                       "_"+str(start)+"_"+str(end)+"_5x5_trend.nc")
    

In [8]:
cmip_files = sorted(glob.glob("../processed_data/CMIP6/pr_day*_5x5.nc"))

for f in cmip_files: 
    ds = xr.open_dataset(f)
    outfile = f.replace("day", "rx1day")
    ## convert to mm from kg/m2/s
    ds["pr"] = ds.pr*60*60*24
    
    ds_rx1day = ds.groupby(ds.time.dt.year).max(dim = "time")
    ds_rx1day.to_netcdf(outfile)
    
    ds_rx1day = ds_rx1day.sel(year = slice(start, end))
    ds_trend = ds_rx1day.polyfit(dim = "year", deg = 1)
    ds_trend.to_netcdf("../processed_data/cmip_trends/"+\
                       f.split("/")[-1].replace("day", "Rx1day").replace(".nc", "_"+str(start)+"_"+str(end)+"_trend.nc"))